# Importing Libraries

In [1]:
 
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy.stats import poisson
from collections import defaultdict
from tqdm import tqdm

In [2]:
base_dir        = Path('../artifacts')
RAW_DATA_DIR = base_dir / 'raw_data'
MODEL_DIR       = base_dir / 'models'
METRICS_DIR     = base_dir / 'metrics'
FIGURE_DIR      = base_dir / 'figures'
SIMULATIONS_DIR = base_dir / 'simulations'
REPORTS_DIR     = base_dir / 'reports'
 
for path in [RAW_DATA_DIR,MODEL_DIR, METRICS_DIR, FIGURE_DIR, SIMULATIONS_DIR, REPORTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)
 

# Load the Data for WorldCup Schedule 

In [3]:
gs_df = pd.read_csv(f'{RAW_DATA_DIR}/group_stages.csv', sep=';')
fk_df = pd.read_csv(f'{RAW_DATA_DIR}/fixtures_knockout_wc2026.csv')

# WC 2026 Official Group Draw 

In [4]:
GROUPS = {}
for group_name, group_data in gs_df.groupby('group'):
    # Sort by position to preserve seeding order
    GROUPS[group_name] = (
        group_data.sort_values('position')['nation'].tolist()
    )


print(f"Groups loaded from CSV: {len(GROUPS)} groups × 4 teams")
for group_name, teams in GROUPS.items():
    print(f"  Group {group_name}: {', '.join(teams)}")

Groups loaded from CSV: 12 groups × 4 teams
  Group A: Mexico, South Africa, South Korea, Czech Republic
  Group B: Canada, Bosnia and Herzegovina, Qatar, Switzerland
  Group C: Brazil, Morocco, Haiti, Scotland
  Group D: United States, Paraguay, Australia, Turkey
  Group E: Germany, Curaçao, Ivory Coast, Ecuador
  Group F: Netherlands, Japan, Sweden, Tunisia
  Group G: Belgium, Egypt, Iran, New Zealand
  Group H: Spain, Cape Verde, Saudi Arabia, Uruguay
  Group I: France, Senegal, Iraq, Norway
  Group J: Argentina, Algeria, Austria, Jordan
  Group K: Portugal, DR Congo, Uzbekistan, Colombia
  Group L: England, Croatia, Ghana, Panama


In [ ]:
HOST_NATIONS = ['United States', 'Canada', 'Mexico']

all_teams = [team for group in GROUPS.values() for team in group]

assert len(all_teams) == 48, f"Expected 48, got {len(all_teams)}"
assert len(set(all_teams)) == 48, "Duplicate teams found!"
print(f"✅ {len(all_teams)} unique teams loaded")

# Loading the Models 

In [ ]:
# Ensemble (stacking meta-model)
meta_model   = joblib.load(f'{MODEL_DIR}/ensemble_meta_model.pkl')
scaler_meta  = joblib.load(f'{MODEL_DIR}/ensemble_meta_scaler.pkl')
 
# Poisson models (needed for ensemble)
poisson_home = joblib.load(f'{MODEL_DIR}/poisson_home.pkl')
poisson_away = joblib.load(f'{MODEL_DIR}/poisson_away.pkl')
poisson_scaler = joblib.load(f'{MODEL_DIR}/poisson_scaler.pkl')
poisson_feature_cols = pd.read_csv(
    f'{base_dir}/features/poisson_feature_cols.csv'
).iloc[:, 0].tolist()
 
# XGBoost models (needed for ensemble)
xgb_home = joblib.load(f'{MODEL_DIR}/xgb_home.pkl')
xgb_away = joblib.load(f'{MODEL_DIR}/xgb_away.pkl')
xgb_feature_cols = pd.read_csv(
    f'{base_dir}/features/xgb_feature_cols.csv'
).iloc[:, 0].tolist()
 
# Dixon-Coles (fallback for unknown teams)
dc_model = joblib.load(f'{MODEL_DIR}/dc_rho.pkl')
rho_fit  = dc_model['rho']
 
# Elo ratings (latest)
df_elo_history = pd.read_csv(f'{base_dir}/processed_data/poisson_features.csv',
                               parse_dates=['date'])
 
print("All models loaded")

All models loaded


# Load Latest Elo Ratings

In [7]:
def get_latest_elo(df_features: pd.DataFrame) -> dict:
    """Extract most recent Elo for each team from feature dataset."""
    latest = (
        df_features.sort_values('date')
        .groupby('home_team')
        .tail(1)[['home_team', 'home_elo']]
        .rename(columns={'home_team': 'team', 'home_elo': 'elo'})
    )
    return latest.set_index('team')['elo'].to_dict()
 
elo_ratings = get_latest_elo(df_elo_history)
 
# Fill missing teams with average
avg_elo = np.mean(list(elo_ratings.values()))
missing = [t for t in all_teams if t not in elo_ratings]
if missing:
    print(f"Teams with no Elo (using avg {avg_elo:.0f}): {missing}")
    for t in missing:
        elo_ratings[t] = avg_elo
 
print(f"\nElo ratings loaded for {len(elo_ratings)} teams")
print("WC 2026 team Elo ratings:")
wc_elos = {t: round(elo_ratings.get(t, avg_elo), 1) for t in all_teams}
for group, teams in GROUPS.items():
    print(f"  Group {group}: " +
          " | ".join(f"{t} ({elo_ratings.get(t, avg_elo):.0f})" for t in teams))


Elo ratings loaded for 314 teams
WC 2026 team Elo ratings:
  Group A: Mexico (1969) | South Africa (1659) | South Korea (1876) | Czech Republic (1777)
  Group B: Canada (1898) | Bosnia and Herzegovina (1643) | Qatar (1565) | Switzerland (1946)
  Group C: Brazil (2068) | Morocco (1983) | Haiti (1714) | Scotland (1819)
  Group D: United States (1832) | Paraguay (1911) | Australia (1880) | Turkey (1954)
  Group E: Germany (1989) | Curaçao (1612) | Ivory Coast (1784) | Ecuador (2021)
  Group F: Netherlands (2000) | Japan (1979) | Sweden (1767) | Tunisia (1751)
  Group G: Belgium (1945) | Egypt (1811) | Iran (1891) | New Zealand (1767)
  Group H: Spain (2221) | Cape Verde (1655) | Saudi Arabia (1685) | Uruguay (1982)
  Group I: France (2119) | Senegal (1927) | Iraq (1744) | Norway (1961)
  Group J: Argentina (2182) | Algeria (1851) | Austria (1880) | Jordan (1777)
  Group K: Portugal (2037) | DR Congo (1771) | Uzbekistan (1824) | Colombia (2051)
  Group L: England (2082) | Croatia (1961) |

# DC Correction Helper 

In [8]:
MAX_GOALS = 10
 
def dc_correction(hg, ag, lambda_h, lambda_a, rho):
    if hg == 0 and ag == 0: return 1 - lambda_h * lambda_a * rho
    if hg == 0 and ag == 1: return 1 + lambda_h * rho
    if hg == 1 and ag == 0: return 1 + lambda_a * rho
    if hg == 1 and ag == 1: return 1 - rho
    return 1.0
 
 
def score_matrix_dc(lambda_h, lambda_a):
    """Build DC-corrected score probability matrix."""
    matrix = np.zeros((MAX_GOALS, MAX_GOALS))
    for hg in range(MAX_GOALS):
        for ag in range(MAX_GOALS):
            p_h = poisson.pmf(hg, lambda_h)
            p_a = poisson.pmf(ag, lambda_a)
            tau = dc_correction(hg, ag, lambda_h, lambda_a, rho_fit)
            matrix[hg, ag] = max(p_h * p_a * tau, 0)
    matrix /= matrix.sum()
    return matrix
 

# Match Prediction Function

In [ ]:
def predict_match(home_team: str, away_team: str,
                  is_neutral: bool = True,
                  stage: str = 'Group Stage') -> dict:
    """
    Predict match probabilities using ensemble.
    Falls back to DC-only if feature data unavailable.
 
    Returns:
        prob_home_win, prob_draw, prob_away_win,
        lambda_home, lambda_away, score_matrix
    """
    home_elo = elo_ratings.get(home_team, avg_elo)
    away_elo = elo_ratings.get(away_team, avg_elo)
    elo_diff = home_elo - away_elo
 
    # ── Try full ensemble prediction ─────────────────────────
    try:
        # Build minimal feature row for Poisson model
        # Using elo_diff and key features available at tournament time
        poisson_feats = {col: 0 for col in poisson_feature_cols}
        poisson_feats['elo_diff']   = elo_diff
        poisson_feats['neutral']    = int(is_neutral)
        poisson_feats['home_elo']   = home_elo
        poisson_feats['away_elo']   = away_elo
 
        # Add any other available features from elo_ratings
        if 'home_is_host' in poisson_feats:
            poisson_feats['home_is_host'] = int(home_team in HOST_NATIONS)
        if 'away_is_host' in poisson_feats:
            poisson_feats['away_is_host'] = int(away_team in HOST_NATIONS)
 
        X_p = pd.DataFrame([poisson_feats])[poisson_feature_cols].fillna(0)
        X_p_scaled = poisson_scaler.transform(X_p)
 
        lh_p = float(poisson_home.predict(X_p_scaled)[0])
        la_p = float(poisson_away.predict(X_p_scaled)[0])
 
        sm_p   = score_matrix_dc(lh_p, la_p)
        p_hw_p = float(np.tril(sm_p, k=-1).sum())
        p_d_p  = float(np.trace(sm_p))
        p_aw_p = float(np.triu(sm_p, k=1).sum())
 
        # XGBoost features
        xgb_feats = {col: 0 for col in xgb_feature_cols}
        xgb_feats['elo_diff']   = elo_diff
        xgb_feats['neutral']    = int(is_neutral)
        xgb_feats['home_elo']   = home_elo
        xgb_feats['away_elo']   = away_elo
 
        X_x = pd.DataFrame([xgb_feats])[xgb_feature_cols].fillna(0)
        lh_x = float(xgb_home.predict(X_x)[0])
        la_x = float(xgb_away.predict(X_x)[0])
 
        sm_x   = score_matrix_dc(lh_x, la_x)
        p_hw_x = float(np.tril(sm_x, k=-1).sum())
        p_d_x  = float(np.trace(sm_x))
        p_aw_x = float(np.triu(sm_x, k=1).sum())
 
        # Stacking ensemble
        X_meta = np.array([[p_hw_p, p_d_p, p_aw_p,
                            p_hw_p, p_d_p, p_aw_p,   # DC ≈ Poisson at inference
                            p_hw_x, p_d_x, p_aw_x]])
        X_meta_scaled = scaler_meta.transform(X_meta)
        probs = meta_model.predict_proba(X_meta_scaled)[0]
 
        lambda_home = (lh_p + lh_x) / 2
        lambda_away = (la_p + la_x) / 2
        score_mat   = score_matrix_dc(lambda_home, lambda_away)
 
        return {
            'prob_home_win': float(probs[0]),
            'prob_draw':     float(probs[1]),
            'prob_away_win': float(probs[2]),
            'lambda_home':   round(lambda_home, 4),
            'lambda_away':   round(lambda_away, 4),
            'score_matrix':  score_mat,
            'method':        'ensemble',
        }
 
    except (KeyError, ValueError):
        # ── DC fallback using Elo-derived λ ──────────────────
        h_adv    = 0.0 if is_neutral else 0.3
        lambda_h = np.exp(0.3 + 0.001 * elo_diff + h_adv)
        lambda_a = np.exp(0.1 - 0.001 * elo_diff)
 
        score_mat  = score_matrix_dc(lambda_h, lambda_a)
        p_home_win = float(np.tril(score_mat, k=-1).sum())
        p_draw     = float(np.trace(score_mat))
        p_away_win = float(np.triu(score_mat, k=1).sum())
 
        return {
            'prob_home_win': p_home_win,
            'prob_draw':     p_draw,
            'prob_away_win': p_away_win,
            'lambda_home':   round(lambda_h, 4),
            'lambda_away':   round(lambda_a, 4),
            'score_matrix':  score_mat,
            'method':        'dc_fallback',
        }

# Simulate a Single Match (stochastic) 

In [10]:
def simulate_match_score(pred: dict) -> tuple[int, int]:
    """
    Sample a scoreline from the DC score probability matrix.
    Returns (home_goals, away_goals).
    """
    sm    = pred['score_matrix']
    flat  = sm.flatten()
    idx   = np.random.choice(len(flat), p=flat / flat.sum())
    hg, ag = divmod(idx, MAX_GOALS)
    return int(hg), int(ag)

# Group Stage Points 

In [11]:
def group_stage_points(goals_for, goals_against):
    if goals_for > goals_against:  return 3, 0
    if goals_for == goals_against: return 1, 1
    return 0, 3

# WC 2026 Format 

In [ ]:
def simulate_tournament(deterministic: bool = False) -> dict:
    """Simulate full WC 2026 using official bracket from fixtures CSV."""

    # ── Group Stage ───────────────────────────────────────────
    group_standings = {}

    for group_name, teams in GROUPS.items():
        stats = {t: {'team': t, 'group': group_name,
                     'points': 0, 'gf': 0, 'ga': 0, 'gd': 0}
                 for t in teams}

        for i in range(len(teams)):
            for j in range(i + 1, len(teams)):
                home, away = teams[i], teams[j]
                pred = predict_match(home, away, is_neutral=True,
                                     stage='Group Stage')

                if deterministic:
                    hg = round(pred['lambda_home'])
                    ag = round(pred['lambda_away'])
                else:
                    hg, ag = simulate_match_score(pred)

                hp, ap = group_stage_points(hg, ag)
                for team, gf, ga, pts in [(home, hg, ag, hp),
                                           (away, ag, hg, ap)]:
                    stats[team]['points'] += pts
                    stats[team]['gf']     += gf
                    stats[team]['ga']     += ga
                    stats[team]['gd']      = (stats[team]['gf'] -
                                              stats[team]['ga'])

        sorted_teams = sorted(
            stats.values(),
            key=lambda x: (x['points'], x['gd'], x['gf']),
            reverse=True
        )
        group_standings[group_name] = sorted_teams

    # ── Get best 3rd-place teams (ranked) ─────────────────────
    third_place = []
    for group_name, standings in group_standings.items():
        third = standings[2].copy()
        third['group'] = group_name
        third_place.append(third)

    best_thirds = sorted(
        third_place,
        key=lambda x: (x['points'], x['gd'], x['gf']),
        reverse=True
    )[:8]   # top 8 advance

    # Track which 3rd-place teams have been used
    used_thirds = set()

    def resolve_slot_inner(slot, match_winners):
        if slot.startswith('W'):
            match_id = 'M' + slot[1:]
            return match_winners.get(match_id)
        if slot.startswith('RU'):
            match_id = 'M' + slot[2:]
            return match_winners.get(f'RU_{match_id}')
        if slot[0].isdigit() and len(slot) >= 2:
            pos   = int(slot[0])
            group = slot[1]
            if pos <= 2:
                return group_standings[group][pos - 1]['team']
        if slot.startswith('3'):
            eligible = list(slot[1:])
            for team_info in best_thirds:
                if (team_info['group'] in eligible and
                        team_info['team'] not in used_thirds):
                    used_thirds.add(team_info['team'])
                    return team_info['team']
        return None

    # ── Knockout Rounds from fixtures CSV ─────────────────────
    match_winners = {}
    round_results = {'group_stage': group_standings}

    round_order = ['R32', 'R16', 'QF', 'SF', '3rd', 'Final']

    for round_name in round_order:
        round_matches = fk_df[fk_df['round'] == round_name].copy()
        round_matchups = []
        round_winners  = []

        for _, fixture in round_matches.iterrows():
            match_id  = fixture['match_id']
            home_slot = fixture['home_slot']
            away_slot = fixture['away_slot']

            home = resolve_slot_inner(home_slot, match_winners)
            away = resolve_slot_inner(away_slot, match_winners)

            if home is None or away is None:
                # Should not happen if bracket logic is correct
                continue

            pred = predict_match(home, away, is_neutral=True,
                                 stage=round_name)

            if deterministic:
                p_hw = pred['prob_home_win']
                p_aw = pred['prob_away_win']
                winner = home if p_hw >= p_aw else away
                loser  = away if p_hw >= p_aw else home   
            else:
                hg, ag = simulate_match_score(pred)
                if hg > ag:
                    winner, loser = home, away
                elif ag > hg:
                    winner, loser = away, home
                else:
                    # Penalties — use win probability
                    p = pred['prob_home_win'] / (
                        pred['prob_home_win'] + pred['prob_away_win']
                    )
                    if np.random.random() < p:
                        winner, loser = home, away
                    else:
                        winner, loser = away, home

            match_winners[match_id]         = winner
            match_winners[f'RU_{match_id}'] = loser

            round_matchups.append((home, away))
            round_winners.append(winner)

        round_results[round_name] = {
            'matchups': round_matchups,
            'winners':  round_winners,
        }

    round_results['champion'] = match_winners.get('M104')
    return round_results

# Deterministic Simulation

In [15]:
print("Running deterministic simulation...")
det_result = simulate_tournament(deterministic=True)
 
print(f"\n Deterministic Result ")
print(f"Champion: {det_result['champion']} 🏆")


print(f"\nQualified for Round of 32:")
if 'R32' in det_result:
    r32_teams = [t for matchup in det_result['R32']['matchups'] for t in matchup]
    for i, team in enumerate(r32_teams, 1):
        print(f"  {i:2d}. {team}")


round_display = {
    'R32':   'Round of 32',
    'R16':   'Round of 16',
    'QF':    'Quarter-finals',
    'SF':    'Semi-finals',
    '3rd':   'Third-place match',
    'Final': 'Final',
}
 
for round_code, round_label in round_display.items():
    if round_code not in det_result:
        continue
    rd = det_result[round_code]
    print(f"\n {round_label} ")
    for (h, a), w in zip(rd['matchups'], rd['winners']):
        print(f"  {h:25s} vs {a:25s}  → {w}")
 
 

Running deterministic simulation...

 Deterministic Result 
Champion: None 🏆

Qualified for Round of 32:
   1. South Korea
   2. Switzerland
   3. Germany
   4. Bosnia and Herzegovina
   5. Netherlands
   6. Morocco
   7. Brazil
   8. Japan
   9. France
  10. Australia
  11. Ecuador
  12. Senegal
  13. Mexico
  14. Ivory Coast
  15. England
  16. Austria
  17. United States
  18. Norway
  19. Belgium
  20. South Africa
  21. Colombia
  22. Croatia
  23. Spain
  24. Algeria
  25. Canada
  26. Egypt
  27. Argentina
  28. Uruguay
  29. Paraguay
  30. Iran

 Round of 32 
  South Korea               vs Switzerland                → Switzerland
  Germany                   vs Bosnia and Herzegovina     → Bosnia and Herzegovina
  Netherlands               vs Morocco                    → Morocco
  Brazil                    vs Japan                      → Japan
  France                    vs Australia                  → Australia
  Ecuador                   vs Senegal                    → Senegal

# Monte Carlo Simulation 

In [16]:
N_SIMULATIONS = 100_000
 
print(f"\nRunning {N_SIMULATIONS:,} Monte Carlo simulations...")
 
# Counters
champion_count    = defaultdict(int)
finalist_count    = defaultdict(int)
semifinal_count   = defaultdict(int)
quarterfinal_count = defaultdict(int)
r16_count         = defaultdict(int)
r32_count         = defaultdict(int)
group_wins_count  = defaultdict(int)
 
for sim in tqdm(range(N_SIMULATIONS), desc='Simulating'):
    result = simulate_tournament(deterministic=False)

    champion_count[result['champion']] += 1

    # Final — both finalists
    if 'Final' in result and result['Final']['matchups']:
        for t in result['Final']['matchups'][0]:
            finalist_count[t] += 1

    # Semi-finals — all 4 teams
    if 'SF' in result:
        for matchup in result['SF']['matchups']:
            for t in matchup:
                semifinal_count[t] += 1

    # Quarter-finals — all 8 teams
    if 'QF' in result:
        for matchup in result['QF']['matchups']:
            for t in matchup:
                quarterfinal_count[t] += 1

    # Round of 16 — all 16 teams
    if 'R16' in result:
        for matchup in result['R16']['matchups']:
            for t in matchup:
                r16_count[t] += 1

    # Round of 32 — all 32 teams
    if 'R32' in result:
        for matchup in result['R32']['matchups']:
            for t in matchup:
                r32_count[t] += 1

    # Group winners
    for group_name, standings in result['group_stage'].items():
        group_wins_count[standings[0]['team']] += 1


Running 100,000 Monte Carlo simulations...
(This will take ~20-30 minutes)



Simulating:   0%|          | 30/100000 [09:07<506:55:37, 18.25s/it]


KeyboardInterrupt: 

# Build Results DataFrame 

In [ ]:
mc_results = pd.DataFrame({'team': all_teams})
 
mc_results['group_win_pct']    = mc_results['team'].map(
    lambda t: group_wins_count.get(t, 0) / N_SIMULATIONS * 100)
mc_results['qualify_r32_pct']  = mc_results['team'].map(
    lambda t: r32_count.get(t, 0) / N_SIMULATIONS * 100)
mc_results['reach_r16_pct']    = mc_results['team'].map(
    lambda t: r16_count.get(t, 0) / N_SIMULATIONS * 100)
mc_results['reach_qf_pct']     = mc_results['team'].map(
    lambda t: quarterfinal_count.get(t, 0) / N_SIMULATIONS * 100)
mc_results['reach_sf_pct']     = mc_results['team'].map(
    lambda t: semifinal_count.get(t, 0) / N_SIMULATIONS * 100)
mc_results['reach_final_pct']  = mc_results['team'].map(
    lambda t: finalist_count.get(t, 0) / N_SIMULATIONS * 100)
mc_results['win_pct']          = mc_results['team'].map(
    lambda t: champion_count.get(t, 0) / N_SIMULATIONS * 100)
 
mc_results = mc_results.sort_values('win_pct', ascending=False).reset_index(drop=True)

print("\n Top 20 Teams  Win Probability ")
print(mc_results.head(20)[[
    'team', 'win_pct', 'reach_final_pct', 'reach_sf_pct',
    'reach_qf_pct', 'reach_r16_pct', 'qualify_r32_pct'
]].to_string(index=False))

# Visualisations 

## Win Probability — Top 20 Teams

In [ ]:
top20 = mc_results.head(20)
 
fig, ax = plt.subplots(figsize=(12, 7))
colors  = ['gold' if i == 0 else 'steelblue' for i in range(len(top20))]
bars    = ax.barh(top20['team'][::-1], top20['win_pct'][::-1],
                  color=colors[::-1], alpha=0.85)
 
for bar, val in zip(bars, top20['win_pct'][::-1]):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=8)
 
ax.set_xlabel('Win Probability (%)')
ax.set_title(f'FIFA World Cup 2026 — Win Probabilities\n'
             f'({N_SIMULATIONS:,} Monte Carlo simulations)', fontsize=12)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/wc2026_win_probabilities.png', dpi=150)
plt.show()

## Tournament Progression Heatmap

In [ ]:
top16     = mc_results.head(16)
prog_cols = ['qualify_r32_pct', 'reach_r16_pct', 'reach_qf_pct',
             'reach_sf_pct', 'reach_final_pct', 'win_pct']
prog_labels = ['Qualify\n(R32)', 'Round\nof 16', 'Quarter\nFinal',
               'Semi\nFinal', 'Final', 'Champion']
 
prog_data = top16.set_index('team')[prog_cols]
 
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(
    prog_data, annot=True, fmt='.1f', cmap='YlOrRd',
    xticklabels=prog_labels, ax=ax,
    linewidths=0.5, cbar_kws={'label': 'Probability (%)'}
)
ax.set_title('WC 2026 — Tournament Progression Probabilities (Top 16 Teams)', fontsize=12)
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/wc2026_progression_heatmap.png', dpi=150)
plt.show()
 

## Group-by-Group Win Probabilities

In [ ]:
n_groups = len(GROUPS)
n_cols   = 4
n_rows   = (n_groups + n_cols - 1) // n_cols
 
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes_flat = axes.flatten()
 
for idx, (group_name, teams) in enumerate(GROUPS.items()):
    ax    = axes_flat[idx]
    g_data = mc_results[mc_results['team'].isin(teams)].copy()
    g_data = g_data.sort_values('qualify_r32_pct', ascending=False)
 
    colors_g = ['gold' if i == 0 else 'steelblue' for i in range(len(g_data))]
    ax.bar(g_data['team'], g_data['qualify_r32_pct'],
           color=colors_g, alpha=0.85)
    ax.set_title(f'Group {group_name}', fontsize=10, fontweight='bold')
    ax.set_ylabel('Qualify %')
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', rotation=15, labelsize=8)
    ax.grid(axis='y', alpha=0.3)
 
    for i, (_, row) in enumerate(g_data.iterrows()):
        ax.text(i, row['qualify_r32_pct'] + 1,
                f"{row['qualify_r32_pct']:.0f}%",
                ha='center', fontsize=7)

for idx in range(len(GROUPS), len(axes_flat)):
    axes_flat[idx].set_visible(False)
 
plt.suptitle('WC 2026 — Group Stage Qualification Probability (%)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/wc2026_group_probabilities.png', dpi=150, bbox_inches='tight')
plt.show()
 


## Champion Probability Distribution

In [ ]:
top10 = mc_results.head(10)
 
fig, ax = plt.subplots(figsize=(10, 5))
colors_c = ['gold'] + ['steelblue'] * 9
 
ax.bar(top10['team'], top10['win_pct'], color=colors_c, alpha=0.85, edgecolor='white')
ax.set_ylabel('Win Probability (%)')
ax.set_title('WC 2026 — Top 10 Favourites to Win', fontsize=12)
for i, (_, row) in enumerate(top10.iterrows()):
    ax.text(i, row['win_pct'] + 0.1, f"{row['win_pct']:.1f}%",
            ha='center', va='bottom', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/wc2026_top10_favourites.png', dpi=150)
plt.show()

## Confidence Intervals

In [ ]:
top15 = mc_results.head(15).copy()
top15['p']     = top15['win_pct'] / 100
top15['ci_95'] = 1.96 * np.sqrt(top15['p'] * (1 - top15['p']) / N_SIMULATIONS) * 100
 
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top15['team'][::-1], top15['win_pct'][::-1],
        xerr=top15['ci_95'][::-1],
        color='steelblue', alpha=0.8, capsize=4, ecolor='black')
 
ax.set_xlabel('Win Probability (%) with 95% CI')
ax.set_title(f'WC 2026 — Win Probabilities with Confidence Intervals\n'
             f'({N_SIMULATIONS:,} simulations)', fontsize=11)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/wc2026_confidence_intervals.png', dpi=150)
plt.show()

# Save Results

In [ ]:
# Full MC results table
mc_results.to_csv(f'{SIMULATIONS_DIR}/wc2026_mc_results.csv', index=False)
 
# Deterministic bracket
det_bracket = {
    'champion':       det_result['champion'],
    'finalists':      list(det_result['Final']['matchups'][0])
                      if det_result.get('Final', {}).get('matchups') else [],
    'semi_finals':    [list(m) for m in det_result.get('SF',  {}).get('matchups', [])],
    'quarter_finals': [list(m) for m in det_result.get('QF',  {}).get('matchups', [])],
    'r16':            [list(m) for m in det_result.get('R16', {}).get('matchups', [])],
    'r32':            [list(m) for m in det_result.get('R32', {}).get('matchups', [])],
}
with open(f'{SIMULATIONS_DIR}/wc2026_deterministic.json', 'w') as f:
    json.dump(det_bracket, f, indent=2)
 
# Summary config
sim_config = {
    'n_simulations':    N_SIMULATIONS,
    'host_nations':     HOST_NATIONS,
    'deterministic_champion': det_result['champion'],
    'mc_top5': mc_results.head(5)[['team', 'win_pct']].to_dict('records'),
}
with open(f'{SIMULATIONS_DIR}/wc2026_config.json', 'w') as f:
    json.dump(sim_config, f, indent=2)
 
print("\nSaved:")
print(f"  simulations/ → wc2026_mc_results.csv")
print(f"  simulations/ → wc2026_deterministic.json")
print(f"  simulations/ → wc2026_config.json")
print(f"  figures/     → wc2026_*.png  (5 plots)")
print(f"\n{'='*60}")
print(f"DETERMINISTIC CHAMPION : {det_result['champion']} 🏆")
print(f"MC TOP 5 FAVOURITES:")
for _, row in mc_results.head(5).iterrows():
    print(f"  {row['team']:25s} {row['win_pct']:.2f}%")
print(f"{'='*60}")